# **(ADD THE NOTEBOOK NAME HERE)**

## Objectives

* Write your notebook objective here, for example, "Fetch data from Kaggle and save as raw data", or "engineer features for modelling"

## Inputs

* Write here which data or information you need to run the notebook 

## Outputs

* Write here which files, code or artefacts you generate by the end of the notebook 

## Additional Comments

* In case you have any additional comments that don't fit in the previous bullets, please state them here. 


---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Users\\PabloGalindo\\Coding-Institute\\PMS5\\heritage-housing-ml-pgz\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Users\\PabloGalindo\\Coding-Institute\\PMS5\\heritage-housing-ml-pgz'

# Step 1: Load Data

Section 1 content

In [4]:
import numpy as np
import pandas as pd
df = (pd.read_csv("outputs/datasets/collection/house_prices_records.csv"))

print(df.shape)
df.head(3)

(1460, 24)


,1stFlrSF,2ndFlrSF,BedroomAbvGr,BsmtExposure,BsmtFinSF1,BsmtFinType1,BsmtUnfSF,EnclosedPorch,GarageArea,GarageFinish,...,LotFrontage,MasVnrArea,OpenPorchSF,OverallCond,OverallQual,TotalBsmtSF,WoodDeckSF,YearBuilt,YearRemodAdd,SalePrice
0,856,854.0,3.0,No,706,GLQ,150,0.0,548,RFn,...,65.0,196.0,61,5,7,856,0.0,2003,2003,208500
1,1262,0.0,3.0,Gd,978,ALQ,284,NaN,460,RFn,...,80.0,0.0,0,8,6,1262,NaN,1976,1976,181500
2,920,866.0,3.0,Mn,486,GLQ,434,0.0,608,RFn,...,68.0,162.0,42,5,7,920,NaN,2001,2002,223500


---

## ML pipeline for Data Cleaning and Feature Engineering

Section 2 content

In [5]:
from sklearn.pipeline import Pipeline
from feature_engine.encoding import OrdinalEncoder
from feature_engine.imputation import (
    MeanMedianImputer,
    CategoricalImputer,
    ArbitraryNumberImputer
)
from feature_engine.transformation import LogTransformer
from feature_engine.selection import DropFeatures, SmartCorrelatedSelection

def PipelineDataCleaningAndFeatureEngineering():
    pipeline_base = Pipeline([

        # 1. Drop irrelevant features (including GarageYrBlt)
        ("DropInitialFeatures", DropFeatures(
            features_to_drop=["WoodDeckSF", "EnclosedPorch", "GarageYrBlt"]
        )),

        # 2. Ordinal Encoding with suffix "_Enc"
        ("OrdinalEncoder", OrdinalEncoder(
            encoding_method="arbitrary",
            variables=["KitchenQual", "GarageFinish", "BsmtExposure", "BsmtFinType1"],
            missing_values='ignore'
        )),

        # 3a. Median Imputation
        ("ImputeMedian", MeanMedianImputer(
            imputation_method="median",
            variables=["2ndFlrSF", "LotFrontage"]
        )),

        # 3b. Mode Imputation
        ("ImputeMode", CategoricalImputer(
            imputation_method="frequent",
            variables=["BedroomAbvGr", "GarageFinish", "BsmtExposure", "BsmtFinType1"],
            ignore_format=True
        )),

        # 3c. Impute 0 for MasVnrArea
        ("ImputeZero", ArbitraryNumberImputer(
            arbitrary_number=0,
            variables=["MasVnrArea"]
        )),

        # 4. Log10 Transformation
        ("LogTransformer", LogTransformer(
            variables=["LotArea", "LotFrontage"],
            base="10"
        )),

        # 5. Drop highly correlated features
        ("SmartCorrelatedSelection", SmartCorrelatedSelection(
            method="spearman",
            threshold=0.7,
            selection_method="variance"
        )),
    ])
    
    return pipeline_base

PipelineDataCleaningAndFeatureEngineering()


Pipeline(steps=[('DropInitialFeatures',
                 DropFeatures(features_to_drop=['WoodDeckSF', 'EnclosedPorch',
                                                'GarageYrBlt'])),
                ('OrdinalEncoder',
                 OrdinalEncoder(encoding_method='arbitrary',
                                missing_values='ignore',
                                variables=['KitchenQual', 'GarageFinish',
                                           'BsmtExposure', 'BsmtFinType1'])),
                ('ImputeMedian',
                 MeanMedianImputer(variables=['2ndFlrSF', 'LotFrontage'])),
                ('I...
                                    variables=['BedroomAbvGr', 'GarageFinish',
                                               'BsmtExposure',
                                               'BsmtFinType1'])),
                ('ImputeZero',
                 ArbitraryNumberImputer(arbitrary_number=0,
                                        variables=['MasVnrArea'])),
                ('LogTransformer',
                 LogTransformer(base='10',
                                variables=['LotArea', 'LotFrontage'])),
                ('SmartCorrelatedSelection',
                 SmartCorrelatedSelection(method='spearman',
                                          selection_method='variance',
                                          threshold=0.7))])

## ML Pipeline for Modelling and Hyperparameter Optimisation

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel

def PipelineReg(model):
    pipeline = Pipeline([
        ("scaler", StandardScaler()), 
        ("feature_selection", SelectFromModel(model, threshold="mean")),  
        ("model", model)  
    ])
    return pipeline

In [7]:
from sklearn.model_selection import GridSearchCV
import numpy as np
import pandas as pd

class HyperparameterOptimizationSearch:

    def __init__(self, models, params):
        self.models = models
        self.params = params
        self.keys = models.keys()
        self.grid_searches = {}

    def fit(self, X, y, cv, n_jobs=-1, verbose=1, scoring='neg_root_mean_squared_error', refit=True):
        for key in self.keys:
            print(f"\n🔍 Running GridSearchCV for {key} \n")
            model = PipelineReg(self.models[key])
            gs = GridSearchCV(
                model,
                self.params[key],
                cv=cv,
                n_jobs=n_jobs,
                verbose=verbose,
                scoring=scoring,
                refit=refit
            )
            gs.fit(X, y)
            self.grid_searches[key] = gs

    def score_summary(self, sort_by='mean_score'):
        def row(key, scores, params):
            return pd.Series({**params, **{
                'estimator': key,
                'min_score': min(scores),
                'max_score': max(scores),
                'mean_score': np.mean(scores),
                'std_score': np.std(scores),
            }})

        rows = []
        for k in self.grid_searches:
            params = self.grid_searches[k].cv_results_['params']
            scores = []
            for i in range(self.grid_searches[k].cv):
                split_scores = self.grid_searches[k].cv_results_[f'split{i}_test_score']
                scores.append(np.array(split_scores).reshape(len(params), 1))
            all_scores = np.hstack(scores)
            for p, s in zip(params, all_scores):
                rows.append(row(k, s, p))

        df = pd.concat(rows, axis=1).T.sort_values([sort_by], ascending=False)
        score_cols = ['estimator', 'min_score', 'mean_score', 'max_score', 'std_score']
        return df[score_cols + [c for c in df.columns if c not in score_cols]], self.grid_searches


### Model + Parameter Grids

In [8]:
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

models = {
    "RandomForest": RandomForestRegressor(random_state=42),
    "XGBoost": XGBRegressor(objective='reg:squarederror', random_state=42),
    "Lasso": Lasso(random_state=42)
}

params = {
    "RandomForest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [10, 20, None]
    },
    "XGBoost": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [3, 6, 10],
        "model__learning_rate": [0.01, 0.1]
    },
    "Lasso": {
        "model__alpha": [0.001, 0.01, 0.1, 1]
    }
}


## Split Train and Test Set

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(['SalePrice'], axis=1),
    df['SalePrice'],
    test_size=0.2,
    random_state=0,
)

print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(1168, 23) (1168,) (292, 23) (292,)


## Modeling the Target Variable: SalePrice


We now have all the necessary preprocessing steps in place to begin modeling the target variable, SalePrice. To start, we will load our custom data cleaning and feature engineering pipeline.

The first model we will test is Linear Regression, which provides a useful performance baseline. After that, we will proceed with hyperparameter optimization to identify the best-performing model for our use case.

The steps are as follows:

- Load and execute the ML pipeline
- Apply a natural logarithm transformation to SalePrice
- Fit and evaluate a Linear Regression model
- Perform hyperparameter optimization to improve model performance

### Executing the ML Pipeline

In [10]:
pipeline_data_cleaning_feat_eng = PipelineDataCleaningAndFeatureEngineering()
X_train = pipeline_data_cleaning_feat_eng.fit_transform(X_train)
X_test = pipeline_data_cleaning_feat_eng.transform(X_test)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(1168, 19) (1168,) (292, 19) (292,)


c:\Users\PabloGalindo\Coding-Institute\PMS5\heritage-housing-ml-pgz\venv\Lib\site-packages\feature_engine\encoding\base_encoder.py:223: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(X[feature]):
c:\Users\PabloGalindo\Coding-Institute\PMS5\heritage-housing-ml-pgz\venv\Lib\site-packages\feature_engine\encoding\base_encoder.py:223: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(X[feature]):
c:\Users\PabloGalindo\Coding-Institute\PMS5\heritage-housing-ml-pgz\venv\Lib\site-packages\feature_engine\encoding\base_encoder.py:223: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(X[feature]):
c:\Users\PabloGalin

### Log Transformation of `SalePrice`

In [34]:
import numpy as np

y_train_log = np.log10(y_train)
y_test_log = np.log10(y_test)

### Execute Linear Regression

In [35]:
from sklearn.linear_model import LinearRegression

linreg = LinearRegression()
linreg.fit(X_train, y_train_log)

y_pred_train_log = linreg.predict(X_train)
y_pred_test_log = linreg.predict(X_test)


### Evaluate Linear Regression

In [36]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate_log_model(y_true_log, y_pred_log, dataset_name=""):
    print(f"--- {dataset_name} (log10 scale) ---")
    print("R2 Score:", r2_score(y_true_log, y_pred_log))
    print("MAE:", mean_absolute_error(y_true_log, y_pred_log))
    print("RMSE:", np.sqrt(mean_squared_error(y_true_log, y_pred_log)))

evaluate_log_model(y_train_log, y_pred_train_log, "Train Set")
evaluate_log_model(y_test_log, y_pred_test_log, "Test Set")


--- Train Set (log10 scale) ---
R2 Score: 0.8835076992862666
MAE: 0.03888764666544878
RMSE: 0.05956206832631707
--- Test Set (log10 scale) ---
R2 Score: 0.689797892166836
MAE: 0.044256892044042484
RMSE: 0.09411728581781599


#### Log-transformation of SalePrice

In [37]:
y_pred_train = np.power(10, y_pred_train_log)
y_pred_test = np.power(10, y_pred_test_log)


In [38]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

def evaluate_original_scale(y_true, y_pred, dataset_name=""):
    print(f"--- {dataset_name} (original scale) ---")
    print("R2 Score:", r2_score(y_true, y_pred))
    print("MAE:", mean_absolute_error(y_true, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_true, y_pred)))

# Evaluate
evaluate_original_scale(y_train, y_pred_train, "Train Set")
evaluate_original_scale(y_test, y_pred_test, "Test Set")


--- Train Set (original scale) ---
R2 Score: 0.7910039257182233
MAE: 16151.462207874587
RMSE: 35871.722968626156
--- Test Set (original scale) ---
R2 Score: -3.5366032643185203
MAE: 26651.88553189741
RMSE: 177000.286693532


### Linear Regression Results – Using `log10(SalePrice)`

To establish a baseline model, we applied a **Linear Regression** algorithm using our custom preprocessing pipeline. The target variable `SalePrice` was transformed using a base-10 logarithm (`log10`) to reduce skewness and stabilize variance.



#### Evaluation on Logarithmic Scale (`log10(SalePrice)`)

| Dataset   | R² Score | MAE   | RMSE  |
|-----------|----------|-------|-------|
| Train Set | 0.88     | 0.039 | 0.060 |
| Test Set  | 0.69     | 0.044 | 0.094 |

- The model performs well on the training set and shows decent generalization on the test set.
- Although the R² score on the test set is slightly below the target of 0.75, the results indicate that the model captures key relationships in the data.



#### Evaluation on Original Scale (After Back-Transformation)

To interpret model performance in actual currency values, we applied the inverse transformation (`10^x`) to the predicted target values.

| Dataset   | R² Score | MAE (€) | RMSE (€) |
|-----------|----------|---------|----------|
| Train Set | 0.79     | 16,151  | 35,872   |
| Test Set  | -3.54    | 26,652  | 177,000  |

- While the training set retains good performance, the test set exhibits a negative R², indicating that predictions are worse than simply predicting the average.
- Therefore, we focus our evaluation on the log-transformed target, which reflects model performance more accurately.


#### Hyperparameter Optimization

Now that we have a baseline model, we will explore additional regression algorithms to identify a model that better predicts our target variable, `SalePrice`.

To do this, we will use our custom hyperparameter optimization class along with dedicated pipelines for each model. The goal is to evaluate and compare the performance of the following three regression models:

- Random Forest Regressor  
- XGBoost Regressor  
- Lasso Regression 



In [ ]:
search = HyperparameterOptimizationSearch(models, params)
search.fit(X_train, y_train_log, cv=5, verbose=2)

summary_df, gs_dict = search.score_summary()
summary_df.head()

---

NOTE

* You may add as many sections as you want, as long as they support your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* If you do not need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create here your folder
  # os.makedirs(name='')
except Exception as e:
  print(e)
